# CIFAR-10 — ResNet Fine-Tuning

## Goal
CIFAR-10 Classification with Pretrained ResNet18

## Setup
- Dataset: CIFAR-10
- Pretrained model: ResNet
- Approach: Replace final layer and train


In [2]:
device = torch.device("cuda")
print(device)

cuda


In [3]:
import torch
import torchvision.models as models
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim

In [4]:
## dataset transformation

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

In [5]:
# loading data

train_data = datasets.CIFAR10('./data', train= True, transform= transform, download= True)
test_data = datasets.CIFAR10('./data', train = False, transform = transform, download = True)

train_loader = DataLoader(train_data, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_data, batch_size= 64, shuffle = False)

100%|██████████| 170M/170M [00:11<00:00, 15.0MB/s]


In [ ]:
## model
model = models.resnet18(pretrained = True)

In [8]:
# freezing the layers weights
for param in model.parameters():
  param.requires_grad = False

In [ ]:
## updating the model last fc layer

model.fc = torch.nn.Linear(512, 10)
model = model.to(device)

In [11]:
optimizer = optim.Adam(model.fc.parameters(), lr= 0.001 )
loss_fn = nn.CrossEntropyLoss()

model.train()

for epoch in range(3):
  total_loss = 0
  avg_loss = 0
  for images, labels in iter(train_loader):
    images = images.to(device)
    labels = labels.to(device)

    out = model(images)
    loss = loss_fn(out, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    avg_loss = total_loss/ len(train_loader)


  print(f'Epoch: {epoch}, Loss = {avg_loss}')



Epoch: 0, Loss = 0.7697716355323792
Epoch: 1, Loss = 0.6178157329559326
Epoch: 2, Loss = 0.5907157063484192


In [16]:
torch.save(model.state_dict(), "resnet18_cifar10_fc_80acc.pth")
print("saved the model")

saved the model


In [13]:
#inference
correct = 0
total = 0
with torch.no_grad():

  model.eval()
  for images, labels in iter(test_loader):
    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    _, preds = outputs.max(dim = 1)

    correct += (preds ==labels).sum().item()
    total += labels.shape[0]

  accuracy = correct/ total

  print("Accuracy is: ", accuracy*100, "%")

Accuracy is:  80.08 %
